# RoBoat Run C training (Kaggle)

This notebook runs the Run C YOLOv8n COCO-control and Zenodo-initialized arms with the same deployment-oriented augmentation policy. It audits the uploaded dataset before training, uses only the combined **validation** split for checkpoint selection, and keeps the combined/source **test** sets and AquaTrash OOD for the final evaluation cell.

## Before running

1. Rebuild the private Run C dataset with `scripts/build_kaggle_bundle.py`; the bundle now includes `audit/manifest.csv`, required for the independent group-leakage check. Upload it as a new Kaggle dataset version.
2. Attach that dataset as this notebook's input and enable a GPU plus Internet (the pinned Python packages are installed below). Keep the dataset private because it contains licensed research images.
3. Run the audit cell. Train the control and Zenodo arms in separate Kaggle sessions if needed. For a resumed session, attach the prior kernel output as an input and set that arm's `*_RESUME_FROM` variable to its run folder.
4. After both arms are complete and the configuration is frozen, run the final test/OOD evaluation cell once. Do not tune based on those final scores.

**Runtime:** each arm is a 100-epoch run on 14,507 training images at 960 px and may take multiple Kaggle sessions. The script resumes from `last.pt` and reuses `best.pt` if the run is already complete.


In [ ]:
%pip install -q ultralytics==8.4.165 albumentations==2.0.8 PyYAML==6.0.3


In [ ]:
# The notebook embeds the matching CLI script, so no separate Kaggle code input is needed.
# All notebook-level imports and path definitions are initialized here.
import base64
import subprocess
import sys
from pathlib import Path

# Fixed directories on Kaggle
DATA_ROOT = Path('/kaggle/input/roboat-run-c')
RUN_ROOT = Path('/kaggle/working/roboat')
SCRIPT = Path('/kaggle/working/train_run_c.py')

SCRIPT_B64 = '''...'''

SCRIPT.write_bytes(base64.b64decode(SCRIPT_B64))
print('Wrote', SCRIPT, '| size:', SCRIPT.stat().st_size, 'bytes')
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT: ', RUN_ROOT)


## 1. Data and augmentation audit

This fails closed on wrong split counts, missing images/labels, malformed or out-of-bounds boxes, exact image duplicates across splits, group leakage in the bundled manifest, source split mismatch, class mapping errors, and AquaTrash overlap with training. It also checks that the domain augmentations preserve a tiny bounding box and writes a visual augmentation preview from a real labeled training image before training starts.


In [ ]:
subprocess.run([sys.executable, str(SCRIPT), "--mode", "audit", "--data-root", str(DATA_ROOT), "--run-root", str(RUN_ROOT)], check=True)


In [ ]:
from IPython.display import display, Image
preview = RUN_ROOT / "augmentation_preview.jpg"
if preview.is_file():
    display(Image(filename=str(preview)))
else:
    print("The real-image augmentation preview is written at the start of the first training arm.")


## 2. Train COCO-control

Custom augmentations model low-probability glare/sun flare, light fog, rain, lower-frame shadows, and mild motion blur. YOLO's native augmentations remain active. No random crop is used, so the pipeline does not discard the already-small targets. Both arms use the same augmentation list, `scale=0.2`, `flipud=0`, seed 42, 960 px, 100 epochs, and patience 20.

To resume from a saved Kaggle output, attach that output as a notebook input and set `CONTROL_RESUME_FROM` to the directory containing `weights/last.pt` (for example `/kaggle/input/<previous-output>/roboat/run_c_control`). Leave it as `None` for a fresh run.


In [ ]:
# Auto-detect previous run checkpoint from kernel_sources or explicit input
CONTROL_RESUME_FROM = None
candidates = [
    Path('/kaggle/input/roboat-run-c-training/roboat/run_c_control'),
    Path('/kaggle/input/roboat-run-c-training/results/roboat/run_c_control'),
    Path('/kaggle/input/roboat-run-c-training/run_c_control'),
    Path('/kaggle/input/previous-output/roboat/run_c_control'),
]
for p in candidates:
    if (p / 'weights' / 'last.pt').is_file() or (p / 'weights' / 'resume_safe.pt').is_file() or (p / 'weights' / 'best.pt').is_file():
        CONTROL_RESUME_FROM = p
        print(f'[resume] Auto-detected previous run checkpoint: {CONTROL_RESUME_FROM}')
        break

cmd = [
    sys.executable, str(SCRIPT),
    '--mode', 'train',
    '--arm', 'control',
    '--data-root', str(DATA_ROOT),
    '--run-root', str(RUN_ROOT),
    '--max-hours', '10.3',
]
if CONTROL_RESUME_FROM is not None:
    cmd += ['--resume', '--resume-from', str(CONTROL_RESUME_FROM)]
else:
    print('[train] Starting fresh control arm training')

print('Executing:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 3. Train Zenodo initialization

This uses the converted `Model_tiles_weights_converted.pt` checkpoint shipped in the same dataset bundle. It uses exactly the same train/validation data and augmentation settings as the COCO-control arm. For a multi-session resume, attach the previous Zenodo kernel output and set `ZENODO_RESUME_FROM` to its `roboat/run_c_zenodo` directory.


In [ ]:
# --- Zenodo arm: run in a SEPARATE Kaggle session after control arm completes ---
# Uncomment ZENODO_RESUME_FROM and the subprocess call when ready to launch the Zenodo arm.
#
# ZENODO_RESUME_FROM = None
# cmd = [
#     sys.executable, str(SCRIPT),
#     '--mode', 'train',
#     '--arm', 'zenodo',
#     '--data-root', str(DATA_ROOT),
#     '--run-root', str(RUN_ROOT),
#     '--max-hours', '10.3',
# ]
# if ZENODO_RESUME_FROM is not None:
#     cmd += ['--resume', '--resume-from', str(ZENODO_RESUME_FROM)]
# subprocess.run(cmd, check=True)


## 4. Final per-source test and AquaTrash OOD evaluation

Run this only after both training arms have finished and the comparison is fixed. It evaluates the predeclared control and Zenodo checkpoints on per-source **test** splits, the combined test split, and the single-class AquaTrash OOD set. Missing evaluation data or a failed source evaluation stops the gate; classes with no ground-truth examples are reported as `N/A`. Results are saved to `final_test_results.json`.

These test/OOD metrics are the final report. Any further augmentation or checkpoint changes require a new untouched evaluation set for an unbiased final score.


In [ ]:
# --- Final evaluation: run ONLY after BOTH arms are complete ---
if False:  # Change to True after both arms finish
    subprocess.run([sys.executable, str(SCRIPT), "--mode", "evaluate", "--arm", "both", "--data-root", str(DATA_ROOT), "--run-root", str(RUN_ROOT)], check=True)


In [ ]:
import json
results_path = RUN_ROOT / "final_test_results.json"
if results_path.is_file():
    print(json.dumps(json.loads(results_path.read_text()), indent=2))
else:
    print("Run the final evaluation cell after both arms are complete.")
